In [6]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

# Updated path to look in the current Colab folder
df = pd.read_excel('Research Survey.xlsx')
df['is_bnpl'] = df['Q4_Used_BNPL'].str.contains('Yes')
bnpl = df[df['is_bnpl']==True]
non_bnpl = df[df['is_bnpl']==False]

likert_map = {'Strongly Disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly Agree': 5}
freq_map = {'Very Low': 1, 'Low': 2, 'Moderate': 3, 'High': 4, 'Very High': 5}
for col in ['Q6a_Compare_Prices','Q6b_Sales_Influence','Q6c_Confident_Without_Trying','Q6d_Comfortable_High_Amounts']:
    df[col+'_num'] = df[col].map(likert_map)
df['shop_freq_num'] = df['Q5a_Shopping_Freq'].map(freq_map)
df['spend_num'] = df['Q5b_Typical_Spend'].map(freq_map)
bnpl = df[df['is_bnpl']==True]
non_bnpl = df[df['is_bnpl']==False]

BLUE = '#1F77B4'
ORANGE = '#FF7F0E'
GRAY = '#AAAAAA'

# FIG 1: Bar chart — mean scores on Q6 items by group
fig, ax = plt.subplots(figsize=(8,5))
labels = ['Compare\nPrices', 'Sales\nInfluence', 'Confident\nWithout Trying', 'Comfortable\nHigh Amounts']
bnpl_means = [bnpl['Q6a_Compare_Prices_num'].mean(), bnpl['Q6b_Sales_Influence_num'].mean(),
              bnpl['Q6c_Confident_Without_Trying_num'].mean(), bnpl['Q6d_Comfortable_High_Amounts_num'].mean()]
nonbnpl_means = [non_bnpl['Q6a_Compare_Prices_num'].mean(), non_bnpl['Q6b_Sales_Influence_num'].mean(),
                 non_bnpl['Q6c_Confident_Without_Trying_num'].mean(), non_bnpl['Q6d_Comfortable_High_Amounts_num'].mean()]
x = np.arange(len(labels))
w = 0.35
bars1 = ax.bar(x-w/2, bnpl_means, w, label='BNPL Users', color=BLUE, edgecolor='white')
bars2 = ax.bar(x+w/2, nonbnpl_means, w, label='Non-BNPL Users', color=ORANGE, edgecolor='white')
ax.set_ylabel('Mean Score (1=Strongly Disagree, 5=Strongly Agree)', fontsize=10)
ax.set_title('Fig. 1: Shopping Behaviour — Mean Scores by Group', fontsize=11, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(1, 5.2)
ax.legend(fontsize=9)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('fig1_group_comparison.png', dpi=150, bbox_inches='tight') # Fixed Path
plt.close()
print("Fig 1 done")

# FIG 2: BNPL group — behavior item means (stacked agreement %)
fig, ax = plt.subplots(figsize=(8,5))
items = ['Increases\nItems Bought', 'Buys More\nExpensive Brands', 'Less Worried\nAbout Spending', 'Shops More\nFrequently', 'Helps Manage\nCash Flow']
cols = ['Q8a_Increases_Items','Q8b_More_Expensive_Brands','Q8c_Less_Worried','Q8d_Shop_More_Freq','Q8e_Manage_Cash_Flow']
agree_pct = []
for col in cols:
    total = bnpl[col].notna().sum()
    agree = ((bnpl[col]=='Agree') | (bnpl[col]=='Strongly Agree')).sum()
    agree_pct.append(agree/total*100)
colors = [BLUE if p >= 50 else ORANGE for p in agree_pct]
bars = ax.barh(items, agree_pct, color=colors, edgecolor='white')
ax.set_xlabel('% of BNPL Users Agreeing or Strongly Agreeing', fontsize=10)
ax.set_title('Fig. 2: BNPL Behavioural Impact — Agreement Rates (BNPL Group Only)', fontsize=11, fontweight='bold')
ax.set_xlim(0, 100)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
for bar, pct in zip(bars, agree_pct):
    ax.text(pct+1, bar.get_y()+bar.get_height()/2, f'{pct:.1f}%', va='center', fontsize=9)
blue_p = mpatches.Patch(color=BLUE, label='>=50% agreement')
orange_p = mpatches.Patch(color=ORANGE, label='<50% agreement')
ax.legend(handles=[blue_p, orange_p], fontsize=9)
plt.tight_layout()
plt.savefig('fig2_bnpl_behavior.png', dpi=150, bbox_inches='tight') # Fixed Path
plt.close()
print("Fig 2 done")

# FIG 3: Monthly spending influence (BNPL group pie)
fig, ax = plt.subplots(figsize=(7,5))
spend_counts = bnpl['Q9_Monthly_Spending_Influence'].value_counts()
short_labels = ['Slightly\nIncreased', 'Significantly\nIncreased', 'Remained\nSame', 'Significantly\nDecreased', 'Slightly\nDecreased']
colors_pie = ['#1F77B4','#2CA02C','#AEC7E8','#FF7F0E','#FFBB78']
wedges, texts, autotexts = ax.pie(spend_counts.values, labels=short_labels, autopct='%1.1f%%',
                                    colors=colors_pie, startangle=90, pctdistance=0.75)
for t in autotexts: t.set_fontsize(9)
for t in texts: t.set_fontsize(9)
ax.set_title('Fig. 3: Self-Reported Monthly Spending Change (BNPL Users, n=125)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_spending_influence.png', dpi=150, bbox_inches='tight') # Fixed Path
plt.close()
print("Fig 3 done")

# FIG 4: Recommend scale distribution (BNPL group)
fig, ax = plt.subplots(figsize=(7,5))
rec = bnpl['Q10_Recommend_Scale_1_10'].value_counts().sort_index()
ax.bar(rec.index.astype(int), rec.values, color=BLUE, edgecolor='white', width=0.7)
ax.set_xlabel('Recommendation Score (1=Unlikely, 10=Very Likely)', fontsize=10)
ax.set_ylabel('Number of Respondents', fontsize=10)
ax.set_title('Fig. 4: BNPL Recommendation Score Distribution (BNPL Users)', fontsize=11, fontweight='bold')
ax.set_xticks(range(5,11))
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
mean_score = bnpl['Q10_Recommend_Scale_1_10'].mean()
ax.axvline(mean_score, color='red', linestyle='--', linewidth=1.5, label=f'Mean = {mean_score:.2f}')
ax.legend(fontsize=9)
for bar in ax.patches:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, str(int(bar.get_height())), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig4_recommend.png', dpi=150, bbox_inches='tight') # Fixed Path
plt.close()
print("Fig 4 done")

# FIG 5: Non-BNPL reasons bar chart
fig, ax = plt.subplots(figsize=(9,5))
reasons_raw = non_bnpl['Q12_Non_BNPL_Reasons'].dropna()
single_reasons = reasons_raw[~reasons_raw.str.contains(',')]
reason_counts = single_reasons.value_counts().head(6)
short = ['Can Afford\nUpfront', 'Too\nComplicated', 'Avoid\nDebt', 'Security\nConcerns', 'No Hard\nInquiries', 'Not\nAware']
ax.barh(short[:len(reason_counts)], reason_counts.values, color=ORANGE, edgecolor='white')
ax.set_xlabel('Number of Respondents', fontsize=10)
ax.set_title('Fig. 5: Reasons for Not Using BNPL (Non-BNPL Group, n=125)', fontsize=11, fontweight='bold')
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
for bar in ax.patches:
    ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2, str(int(bar.get_width())), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig5_nonbnpl_reasons.png', dpi=150, bbox_inches='tight') # Fixed Path
plt.close()
print("Fig 5 done")

Fig 1 done
Fig 2 done
Fig 3 done
Fig 4 done
Fig 5 done
